# Lesson 4 — State and memory

**You will:** distinguish short-term memory (the messages list) from long-term state (the `State` dict), and persist information across runs.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/04-state-and-memory/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

LLMs have no memory of their own. Every call is fresh. The only way an agent 'remembers' is because *you* hand it the same messages back next turn.

Within one `bear.run()`, the messages list grows turn by turn — that's short-term memory. Across runs, you need something else: an explicit, inspectable, snapshotable dict that the agent can read and write. BareBear calls it `State`.

**Most 'magic memory' features in agent frameworks are just a dict.** Once you've seen the dict, the magic disappears.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")

# Optional: pin a specific free-tier model. If unset, barebear uses its default.
# Free-tier model availability rotates — swap here if a lesson cell errors with
# "model not available". Any *:free model on https://openrouter.ai/models works.
# os.environ["BAREBEAR_MODEL"] = "meta-llama/llama-3.2-3b-instruct:free"

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")
print("Model:", os.environ.get("BAREBEAR_MODEL", "(framework default)"))

## A note-taker that survives between conversations

In [ ]:
from barebear import Bear, Task, Tool, State, OpenRouterModel

state = State({"notes": []})

def add_note(text: str) -> str:
    state.set("notes", state.get("notes") + [text])
    return f"Stored. {len(state.get('notes'))} note(s) total."

def list_notes() -> str:
    notes = state.get("notes")
    if not notes:
        return "No notes."
    return "\n".join(f"- {n}" for n in notes)

tools = [
    Tool(name="add_note", fn=add_note, description="Add a note to long-term memory."),
    Tool(name="list_notes", fn=list_notes, description="List all stored notes."),
]

bear = Bear(model=OpenRouterModel(), tools=tools, state=state)

print("--- run 1: tell it something to remember ---")
bear.run(Task(goal="Remember that I like Earl Grey tea."))

print("\n--- run 2: a fresh agent loop, but state persists ---")
bear.run(Task(goal="What do I like to drink?"), trace=True)

print("\n--- final state ---")
print(state.snapshot())

## What just happened

The second `bear.run()` is a totally fresh agent loop with no message history from the first run. But the *state* persisted. So the agent could call `list_notes()` and find what you told it before.

This is the entire pattern. Long-term memory in agents is just: store things in a dict, give the agent a tool to read/write it.

## Exercise

1. Add a `delete_note(index)` tool. Verify the agent uses it when asked.
2. Use `state.snapshot()` between runs and look at the diff. Read [`src/barebear/state.py`](https://github.com/richey-malhotra/barebear/blob/main/src/barebear/state.py) — about 60 lines.

In [ ]:
def delete_note(index: int) -> str:
    notes = state.get("notes")
    if 0 <= index < len(notes):
        removed = notes.pop(index)
        state.set("notes", notes)
        return f"Removed: {removed}"
    return f"No note at index {index}."

# Build a new bear with the delete tool and try it.

## What's next

[Lesson 5 →](../05-budgets/lesson.ipynb)